In [ ]:
# ==========================================
# 0. УСТАНОВКА ЗАВИСИМОСТЕЙ (для нового блокнота)
# ==========================================
!pip install django Pillow

import os
import time
import threading
from google.colab.output import eval_js

# ==========================================
# 1. СОЗДАНИЕ ПРОЕКТА
# ==========================================
%cd /content
!django-admin startproject myshop
%cd /content/myshop
!python manage.py startapp shop

# ==========================================
# 2. НАСТРОЙКА SETTINGS.PY
# ==========================================
filepath = '/content/myshop/myshop/settings.py'
with open(filepath, 'r') as f:
    code = f.read()

code = code.replace(
    "'django.contrib.staticfiles',",
    "'django.contrib.staticfiles',\n    'shop',"
)

code += """
import os
STATIC_URL = '/static/'
STATICFILES_DIRS = [os.path.join(BASE_DIR, 'shop/static')]
MEDIA_URL = '/media/'
MEDIA_ROOT = os.path.join(BASE_DIR, 'media')
CART_SESSION_ID = 'cart'

# Для работы в Colab
ALLOWED_HOSTS = ['*']
CSRF_TRUSTED_ORIGINS = ['https://*.colab.dev']
SECURE_PROXY_SSL_HEADER = ('HTTP_X_FORWARDED_PROTO', 'https')
"""
with open(filepath, 'w') as f:
    f.write(code)

# ==========================================
# 3. МОДЕЛИ И АДМИНКА
# ==========================================
models_code = """
from django.db import models
from django.urls import reverse

class Category(models.Model):
    name = models.CharField(max_length=200, verbose_name="Название")
    slug = models.SlugField(max_length=200, unique=True, verbose_name="URL")
    class Meta:
        verbose_name = "Категория"
        verbose_name_plural = "Категории"
    def __str__(self): return self.name
    def get_absolute_url(self): return reverse('shop:product_list_by_category', args=[self.slug])

class Product(models.Model):
    category = models.ForeignKey(Category, related_name='products', on_delete=models.CASCADE, verbose_name="Категория")
    name = models.CharField(max_length=200, verbose_name="Название")
    slug = models.SlugField(max_length=200, unique=True, verbose_name="URL")
    size = models.CharField(max_length=50, verbose_name="Размер")
    material = models.CharField(max_length=50, verbose_name="Материал")
    price = models.DecimalField(max_digits=10, decimal_places=2, verbose_name="Цена")
    stock = models.PositiveIntegerField(default=0, verbose_name="Остаток")
    image = models.ImageField(upload_to='products/', blank=True, null=True, verbose_name="Фото")
    available = models.BooleanField(default=True, verbose_name="В наличии")
    class Meta:
        verbose_name = "Товар"
        verbose_name_plural = "Товары"
    def __str__(self): return self.name
    def get_absolute_url(self): return reverse('shop:product_detail', args=[self.id, self.slug])

class Order(models.Model):
    first_name = models.CharField(max_length=50, verbose_name="Имя")
    last_name = models.CharField(max_length=50, verbose_name="Фамилия")
    email = models.EmailField(verbose_name="Email")
    address = models.CharField(max_length=250, verbose_name="Адрес")
    created = models.DateTimeField(auto_now_add=True, verbose_name="Дата")
    paid = models.BooleanField(default=False, verbose_name="Оплачен")
    class Meta:
        verbose_name = "Заказ"
        verbose_name_plural = "Заказы"
        ordering = ('-created',)
    def __str__(self): return f'Заказ {self.id}'

class OrderItem(models.Model):
    order = models.ForeignKey(Order, related_name='items', on_delete=models.CASCADE)
    product = models.ForeignKey(Product, related_name='order_items', on_delete=models.CASCADE)
    price = models.DecimalField(max_digits=10, decimal_places=2)
    quantity = models.PositiveIntegerField(default=1)
    def __str__(self): return str(self.id)
"""
admin_code = """
from django.contrib import admin
from .models import Category, Product, Order, OrderItem

@admin.register(Category)
class CategoryAdmin(admin.ModelAdmin):
    list_display = ['name', 'slug']
    prepopulated_fields = {'slug': ('name',)}

@admin.register(Product)
class ProductAdmin(admin.ModelAdmin):
    list_display = ['name', 'category', 'price', 'stock', 'available']
    list_filter = ['available', 'category', 'material']
    list_editable = ['price', 'stock', 'available']
    search_fields = ['name', 'size', 'material']
    prepopulated_fields = {'slug': ('name',)}

class OrderItemInline(admin.TabularInline):
    model = OrderItem
    raw_id_fields = ['product']

@admin.register(Order)
class OrderAdmin(admin.ModelAdmin):
    list_display = ['id', 'first_name', 'last_name', 'email', 'paid', 'created']
    list_filter = ['paid', 'created']
    inlines = [OrderItemInline]
"""
with open('/content/myshop/shop/models.py', 'w') as f: f.write(models_code)
with open('/content/myshop/shop/admin.py', 'w') as f: f.write(admin_code)

# ==========================================
# 4. КОРЗИНА И ФОРМЫ
# ==========================================
cart_code = """
from decimal import Decimal
from django.conf import settings
from .models import Product
class Cart:
    def __init__(self, request):
        self.session = request.session
        cart = self.session.get(settings.CART_SESSION_ID)
        if not cart:
            cart = self.session[settings.CART_SESSION_ID] = {}
        self.cart = cart
    def add(self, product, quantity=1, update_quantity=False):
        product_id = str(product.id)
        if product_id not in self.cart:
            self.cart[product_id] = {'quantity': 0, 'price': str(product.price)}
        if update_quantity: self.cart[product_id]['quantity'] = quantity
        else: self.cart[product_id]['quantity'] += quantity
        self.save()
    def remove(self, product):
        product_id = str(product.id)
        if product_id in self.cart:
            del self.cart[product_id]
            self.save()
    def save(self):
        self.session[settings.CART_SESSION_ID] = self.cart
        self.session.modified = True
    def __iter__(self):
        product_ids = self.cart.keys()
        products = Product.objects.filter(id__in=product_ids)
        for product in products:
            self.cart[str(product.id)]['product'] = product
        for item in self.cart.values():
            item['price'] = Decimal(item['price'])
            item['total_price'] = item['price'] * item['quantity']
            yield item
    def __len__(self): return sum(item['quantity'] for item in self.cart.values())
    def get_total_price(self): return sum(Decimal(item['price']) * item['quantity'] for item in self.cart.values())
    def clear(self):
        del self.session[settings.CART_SESSION_ID]
        self.session.modified = True
"""
forms_code = """
from django import forms
from .models import Order
class OrderCreateForm(forms.ModelForm):
    class Meta:
        model = Order
        fields = ['first_name', 'last_name', 'email', 'address']
"""
with open('/content/myshop/shop/cart.py', 'w') as f: f.write(cart_code)
with open('/content/myshop/shop/forms.py', 'w') as f: f.write(forms_code)

# ==========================================
# 5. ПРЕДСТАВЛЕНИЯ И МАРШРУТЫ
# ==========================================
views_code = """
from django.shortcuts import render, get_object_or_404, redirect
from django.views.decorators.http import require_POST
from .models import Category, Product
from .forms import OrderCreateForm
from .cart import Cart

def product_list(request, category_slug=None):
    category = None
    categories = Category.objects.all()
    products = Product.objects.filter(available=True)
    if category_slug:
        category = get_object_or_404(Category, slug=category_slug)
        products = products.filter(category=category)
    return render(request, 'shop/product_list.html', {'category': category, 'categories': categories, 'products': products})

def product_detail(request, id, slug):
    product = get_object_or_404(Product, id=id, slug=slug, available=True)
    return render(request, 'shop/product_detail.html', {'product': product})

@require_POST
def cart_add(request, product_id):
    cart = Cart(request)
    product = get_object_or_404(Product, id=product_id)
    cart.add(product=product, quantity=int(request.POST.get('quantity', 1)))
    return redirect('shop:cart_detail')

def cart_remove(request, product_id):
    cart = Cart(request)
    product = get_object_or_404(Product, id=product_id)
    cart.remove(product)
    return redirect('shop:cart_detail')

def cart_detail(request):
    cart = Cart(request)
    return render(request, 'shop/cart.html', {'cart': cart})

def order_create(request):
    cart = Cart(request)
    if request.method == 'POST':
        form = OrderCreateForm(request.POST)
        if form.is_valid():
            order = form.save()
            for item in cart:
                order.items.create(product=item['product'], price=item['price'], quantity=item['quantity'])
            cart.clear()
            return render(request, 'shop/order_success.html', {'order': order})
    else:
        form = OrderCreateForm()
    return render(request, 'shop/checkout.html', {'cart': cart, 'form': form})
"""
urls_code = """
from django.urls import path
from . import views
app_name = 'shop'
urlpatterns = [
    path('', views.product_list, name='product_list'),
    path('cart/', views.cart_detail, name='cart_detail'),
    path('cart/add/<int:product_id>/', views.cart_add, name='cart_add'),
    path('cart/remove/<int:product_id>/', views.cart_remove, name='cart_remove'),
    path('checkout/', views.order_create, name='order_create'),
    path('<int:id>/<slug:slug>/', views.product_detail, name='product_detail'),
]
"""
main_urls = """
from django.contrib import admin
from django.urls import path, include
from django.conf import settings
from django.conf.urls.static import static
urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('shop.urls')),
]
if settings.DEBUG:
    urlpatterns += static(settings.MEDIA_URL, document_root=settings.MEDIA_ROOT)
"""
with open('/content/myshop/shop/views.py', 'w') as f: f.write(views_code)
with open('/content/myshop/shop/urls.py', 'w') as f: f.write(urls_code)
with open('/content/myshop/myshop/urls.py', 'w') as f: f.write(main_urls)

# ==========================================
# 6. ШАБЛОНЫ
# ==========================================
os.makedirs('/content/myshop/shop/templates/shop', exist_ok=True)
base = """{% load static %}<!DOCTYPE html><html><head><title>Магазин</title><link rel="stylesheet" href="{% static 'shop/css/style.css' %}"></head><body><header><h1><a href="/">Шурупы</a></h1><a href="{% url 'shop:cart_detail' %}">Корзина</a></header><main>{% block content %}{% endblock %}</main></body></html>"""
plist = """{% extends 'shop/base.html' %}{% block content %}<h2>Каталог</h2><div class="products">{% for product in products %}<div class="product"><h3><a href="{{ product.get_absolute_url }}">{{ product.name }}</a></h3><p>{{ product.price }} руб.</p><form action="{% url 'shop:cart_add' product.id %}" method="post">{% csrf_token %}<button type="submit">Купить</button></form></div>{% endfor %}</div>{% endblock %}"""
pdetail = """{% extends 'shop/base.html' %}{% block content %}<h2>{{ product.name }}</h2><p>Размер: {{ product.size }}</p><p>Цена: {{ product.price }} руб.</p><form action="{% url 'shop:cart_add' product.id %}" method="post">{% csrf_token %}<input type="number" name="quantity" value="1" min="1"><button type="submit">Добавить</button></form>{% endblock %}"""
cart_t = """{% extends 'shop/base.html' %}{% block content %}<h2>Корзина</h2><table>{% for item in cart %}<tr><td>{{ item.product.name }}</td><td>{{ item.quantity }} шт.</td><td>{{ item.total_price }} руб.</td><td><a href="{% url 'shop:cart_remove' item.product.id %}">Удалить</a></td></tr>{% endfor %}</table><p>Итого: {{ cart.get_total_price }} руб.</p><a href="{% url 'shop:order_create' %}">Оформить</a>{% endblock %}"""
checkout_t = """{% extends 'shop/base.html' %}{% block content %}<h2>Оформление</h2><form method="post">{% csrf_token %}{{ form.as_p }}<button type="submit">Отправить</button></form>{% endblock %}"""
success_t = """{% extends 'shop/base.html' %}{% block content %}<h2>Заказ {{ order.id }} оформлен!</h2>{% endblock %}"""

for name, content in [('base.html', base), ('product_list.html', plist), ('product_detail.html', pdetail), ('cart.html', cart_t), ('checkout.html', checkout_t), ('order_success.html', success_t)]:
    with open(f'/content/myshop/shop/templates/shop/{name}', 'w') as f: f.write(content)

os.makedirs('/content/myshop/shop/static/shop/css', exist_ok=True)
with open('/content/myshop/shop/static/shop/css/style.css', 'w') as f: f.write("body { font-family: sans-serif; padding: 20px; } .product { border: 1px solid #ccc; padding: 10px; margin-bottom: 10px; }")

# ==========================================
# 7. МИГРАЦИИ И ЗАПУСК
# ==========================================
!python manage.py makemigrations
!python manage.py migrate

os.environ['DJANGO_SUPERUSER_PASSWORD'] = 'admin12345'
!python manage.py createsuperuser --noinput --username admin --email admin@example.com

os.system('pkill -f runserver')
time.sleep(1)

def run_django():
    os.system('python manage.py runserver 0.0.0.0:8000')

threading.Thread(target=run_django).start()
time.sleep(5)

print("\n✅ ПРОЕКТ ГОТОВ! ВАША ССЫЛКА:")
print(eval_js("google.colab.kernel.proxyPort(8000)"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 74.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [django]
/content
/content/myshop
Migrations for 'shop':
  shop/migrations/0001_initial.py
    + Create model Category
    + Create model Order
    + Create model Product
    + Create model OrderItem
Operations to perform:
  Apply all migrations: admin, auth, contenttypes, sessions, shop
Running migrations:
  Applying contenttypes.0001_initial... OK
  Applying auth.0001_initial... OK
  Applying admin.0001_initial... OK
  Applying admin.0002_logentry_remove_auto_add... OK
  Applying admin.0003_logentry_add_action_flag_choices... OK
  Applying contenttypes.0002_remove_content_type_name... OK
  Applying auth.0002_alter_permission_name_max_length... OK
  Applying auth.0003_alter_user_email_max_length... OK
  Applying auth.0004_alter_user_username_opts... OK
  Applying auth.0005_alter_user_last_login_null... OK
  Applying auth.0006_require_contenttypes_0002... OK
  Apply

In [ ]:
# Читаем текущий шаблон
filepath = '/content/myshop/shop/templates/shop/product_list.html'
with open(filepath, 'r') as f:
    code = f.read()

# Добавляем вывод картинки, если она есть
new_img_code = """{% if product.image %}
    <img src="{{ product.image.url }}" alt="{{ product.name }}" style="width: 100px; height: 100px; object-fit: contain;"><br>
{% endif %}"""

# Вставляем картинку перед названием
code = code.replace('<h3>', new_img_code + '<h3>')

with open(filepath, 'w') as f:
    f.write(code)

print("Картинки добавлены в шаблон! Теперь просто обновите страницу сайта в браузере.")

Картинки добавлены в шаблон! Теперь просто обновите страницу сайта в браузере.
